# Predicción de Ingresos — Adult (versión de práctica)

**Proyecto sustituto para la Tarea 4.1.** El tercer proyecto del aula virtual no estaba disponible al momento de la auditoría; este notebook reproduce la estructura del problema *Adult Income* del curso (predecir si una persona gana más de 50K) con **datos sintéticos generados con semilla fija**. Se escribió como lo haría un equipo con prisa: no se corrigió nada antes de auditarlo.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rng = np.random.RandomState(7)

## 1. Datos

In [2]:
n = 4000
df = pd.DataFrame({
    'age': rng.randint(18, 70, n),
    'education_num': rng.randint(3, 17, n),
    'hours_per_week': np.clip(rng.normal(40, 11, n), 5, 90).round(),
    'sex': rng.choice(['Male', 'Female'], n, p=[0.67, 0.33]),
})
df['capital_gain'] = np.where(rng.rand(n) < 0.08, rng.lognormal(8.5, 1.0, n), 0).round()
logit = (-9.2 + 0.05 * df['age'] + 0.33 * df['education_num'] + 0.035 * df['hours_per_week']
         + 0.0004 * df['capital_gain'] + 0.9 * (df['sex'] == 'Male'))
df['income_gt_50k'] = (rng.rand(n) < 1 / (1 + np.exp(-logit))).astype(int)
df['sex_male'] = (df['sex'] == 'Male').astype(int)
print(df['income_gt_50k'].value_counts(normalize=True).round(3))
df.head()

income_gt_50k
0    0.749
1    0.251
Name: proportion, dtype: float64


   age  education_num  hours_per_week  ... capital_gain  income_gt_50k  sex_male
0   65             11            50.0  ...          0.0              1         1
1   22             13            26.0  ...          0.0              0         0
2   43              9            37.0  ...          0.0              1         1
3   21              4            52.0  ...          0.0              0         0
4   37             10            43.0  ...          0.0              0         1

[5 rows x 7 columns]

## 2. Preparación

Escalamos las variables numéricas para que la regresión logística converja mejor.

In [3]:
PREDICTORES = ['age', 'education_num', 'hours_per_week', 'capital_gain', 'sex_male']
X = df[PREDICTORES]
y = df['income_gt_50k']

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

## 3. Modelos

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X_escalado, y, test_size=0.2, random_state=1)
logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
acc_logreg = accuracy_score(y_test, logreg.predict(X_test))

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_escalado, y, test_size=0.2)
rf = RandomForestClassifier(n_estimators=200).fit(X_train2, y_train2)
acc_rf = accuracy_score(y_test2, rf.predict(X_test2))

print(f"Accuracy regresión logística: {acc_logreg:.3f}")
print(f"Accuracy random forest:       {acc_rf:.3f}")

Accuracy regresión logística: 0.809
Accuracy random forest:       0.796


## 4. Conclusión

Ambos modelos superan el 75 % de accuracy, así que el modelo es apto para usarse en la preselección de solicitudes. Elegimos el de mayor accuracy.